In [29]:
# =============================================================================
# CELL 1: CONFIGURATION, BASELINES, POOL DEFINITIONS
# =============================================================================

granularity = 'm'
run_every_query = False

LOBS = ['STG']

BASELINES = {
    'STG': {'ltv': 1.94, 'recovery_pct': 0.4308,
            'recovery_unadjusted_pct': 0.4308, 'apr': 0.25},
}

MODEL_PARAMS = {
    'mean_unit_loss': 0.5,
    'unit_loss_to_model_score': 0.02,
    'skip_rate': 0.75,
    'expected_years_on_book': 2,
    'impound_probability': 0.15,
    'mmi_standard_increase': 1.03,
}

MODEL_SCORE_THRESHOLD = 140

POOLS = {
    'Vanilla':    {'filter': None},
    'Downcredit': {'filter': lambda df: df[df['cd_model_score'] < MODEL_SCORE_THRESHOLD]},
    'Upcredit':   {'filter': lambda df: df[df['cd_model_score'] >= MODEL_SCORE_THRESHOLD]},
}

print(f"Pools: {list(POOLS.keys())}")
print(f"Model score threshold: {MODEL_SCORE_THRESHOLD}")

Pools: ['Vanilla', 'Downcredit', 'Upcredit']
Model score threshold: 140


In [30]:
# =============================================================================
# CELL 2: IMPORTS + DERIVED CONFIGURATION
# =============================================================================

import pandas as pd
import numpy as np
import pyodbc
import pickle
import warnings
import os
import datetime as dt
import openpyxl

pd.set_option('display.max_columns', 100)
pd.set_option('display.min_rows', 100)

max_month = 12
max_year = dt.date.today().year

mean_unit_loss = MODEL_PARAMS['mean_unit_loss']
unit_loss_to_model_score = MODEL_PARAMS['unit_loss_to_model_score']
skip_rate = MODEL_PARAMS['skip_rate']
expected_years_on_book = MODEL_PARAMS['expected_years_on_book']
impound_probability = MODEL_PARAMS['impound_probability']
mmi_standard_increase = MODEL_PARAMS['mmi_standard_increase']

print(f"Granularity: {granularity}")
print(f"Max year: {max_year}, max month: {max_month}")

Granularity: m
Max year: 2026, max month: 12


In [31]:
# =============================================================================
# CELL 3: UTILITY FUNCTIONS
# =============================================================================

def run_sql(filename, sub_list=None, connection=None):
    if sub_list is None:
        sub_list = []
    with open(filename, 'r') as file:
        query = file.read()
    for text, var in sub_list:
        query = query.replace(text, var)
    if connection is None:
        with pyodbc.connect("DSN=Redshift_prod_new") as conn:
            warnings.filterwarnings("ignore", category=UserWarning)
            df = pd.read_sql_query(sql=query, con=conn)
            warnings.filterwarnings("default", category=UserWarning)
            return df
    else:
        warnings.filterwarnings("ignore", category=UserWarning)
        df = pd.read_sql_query(sql=query, con=connection)
        warnings.filterwarnings("default", category=UserWarning)
        return df


def store_pickle(data, filename):
    if type(data) == str:
        data, filename = filename, data
    with open(filename, 'wb') as file:
        pickle.dump(data, file)


def get_pickle(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)


def cached_sql(query_file, pickle_name, connection=None, force_refresh=False):
    if force_refresh or not os.path.exists(pickle_name):
        df = run_sql(query_file, connection=connection)
        if df.empty:
            print(f"  WARNING: query returned 0 rows")
        store_pickle(df, pickle_name)
        return df
    df = get_pickle(pickle_name)
    if df.empty:
        print(f"  WARNING: cached '{pickle_name}' contains 0 rows")
    return df


def weight_by_proceeds(metric, proceeds):
    return (metric * proceeds).sum() / proceeds.sum()


def weighted_average_and_sum(group, metrics, include_groups=False):
    if type(metrics) == str:
        weighted_avg = (group[metrics] * group.amt_financed).sum() / group.amt_financed.sum()
        return pd.Series({metrics: weighted_avg, 'amt_financed': group.amt_financed.sum()})
    result_dict = {'amt_financed': group.amt_financed.sum()}
    for metric in metrics:
        weighted_avg = (group[metric] * group.amt_financed).sum() / group.amt_financed.sum()
        result_dict[metric] = weighted_avg
    return pd.Series(result_dict)


print('Utility functions loaded')

Utility functions loaded


In [32]:
# =============================================================================
# CELL 4: ULA MULTIPLIER + RECOVERY MODEL (auc_pred)
# =============================================================================

def get_ula_multiplier_nonkmx(ula_df, leave_out):
    ula_df['loss_multiplier'] = 1
    if leave_out != 'Previous ACA chargeoff':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.prev_co_flag
    if leave_out != 'Small amount financed':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.small_amt_financed_flag * (1 - ula_df.pricing_change_flag)
    if leave_out != 'Zero cash down':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.zero_cash_down_flag * (1 - ula_df.pricing_change_flag)
    if leave_out != 'High mileage vehicle':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_mileage_vehicle_flag
    if leave_out != 'High PTI':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_pti_flag * (1 - ula_df.pricing_change_flag)
    if leave_out != 'Car make':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.car_make_penalty_flag \
                                    - 0.1 * ula_df.car_make_benefit_flag \
                                    - 0.1 * ula_df.pricing_change_flag * ula_df.car_make_benefit_flag
    if leave_out != 'Theft risk':
        ula_df.loss_multiplier *= 0.987 + 0.099 * ula_df.theft_risk_flag * (1 - ula_df.pricing_change_flag) + 0.013 * ula_df.pricing_change_flag
    if leave_out != 'MCY high model score, low mileage':
        ula_df.loss_multiplier *= 1 - 0.2 * ula_df.mcy_low_mileage_flag
    if leave_out != 'Weekday/weekend decision':
        ula_df.loss_multiplier *= 1 - 0.05 * ula_df.weekend_flag + 0.02 * ula_df.weekday_flag
    if leave_out != 'Student Loans':
        ula_df.loss_multiplier *= 1 + (0.1 * ula_df.student_loan_flag \
                                       - np.minimum(np.maximum((ula_df.cd_model_score - (130 - 2 * ula_df.ent_fld_flag)) * 0.006, 0), 0.03) \
                                       * (1 - ula_df.student_loan_flag)) * (ula_df.student_loans_cutoff_date)
    if leave_out != 'Low PTI':
        ula_df.loss_multiplier *= 1 + (-0.03 * np.minimum(ula_df.cd_model_score, 135) + 3.9) * ula_df.low_pti_flag
    if leave_out != 'Fraud':
        ula_df.loss_multiplier *= 1 + (ula_df.fraud_adjustment - 1)
    if leave_out != 'Driver flag':
        ula_df.loss_multiplier *= 1 + 0.15 * ula_df.driver_flag
    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loss_multiplier *= 0.966 + 0.273 * ula_df.nonkmx_chime_flag
    if leave_out != 'Employment type':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.seasonal_employment_flag \
                                    - 0.1 * ula_df.waiter_employment_flag
    if leave_out != 'Authorized tradelines':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.nonkmx_auth_tradelines_flag
    if leave_out != 'Clip':
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.8, 2)
    if leave_out == 'Dealer Level (Non-KMX)':
        ula_df.loss_multiplier = ula_df.loss_multiplier * ula_df.pricing_scalar
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.7, 1.4)
    return ula_df


def auc_pred(rra_df, leave_out, rra_lob_df=None):
    if len(rra_df[rra_df.lob == 'KMX']) > 0:
        conditions = [
            (rra_df.car_age_orig < 3),
            (rra_df.car_age_orig == 3) & (rra_df.sale_price < 30000),
            (rra_df.car_age_orig == 3) & (30000 <= rra_df.sale_price) & (rra_df.sale_price < 40000),
            (rra_df.car_age_orig == 3) & (rra_df.sale_price >= 40000),
            (3 < rra_df.car_age_orig) & (rra_df.car_age_orig < 6) & (rra_df.sale_price < 30000),
            (3 < rra_df.car_age_orig) & (rra_df.car_age_orig < 6) & (30000 <= rra_df.sale_price) & (rra_df.sale_price < 40000),
            (3 < rra_df.car_age_orig) & (rra_df.car_age_orig < 6) & (rra_df.sale_price >= 40000),
            (rra_df.car_age_orig >= 6) & (rra_df.sale_price < 30000),
            (rra_df.car_age_orig >= 6) & (30000 <= rra_df.sale_price) & (rra_df.sale_price < 40000),
            (rra_df.car_age_orig >= 6) & (rra_df.sale_price >= 40000)
        ]
        values = [3300, 3700, 5400, 7200, 4700, 6400, 8200, 5700, 7400, 9200]
        recovery_amount = np.maximum(0, rra_df.sale_price - np.select(conditions, values, default=0))
        return recovery_amount * (1 - 0.206) ** rra_df.yob / rra_df.sale_price

    rra_df['coef_intercept'] = 0.0891185245
    rra_df['coef_yob'] = -0.2668087106 * rra_df.yob
    if leave_out != 'Mileage/age':
        rra_df['coef_mlg_x_age'] = 0.0001463508 * rra_df.car_age_orig * rra_df.mileage - 0.0004310013 * rra_df.car_age_orig - 0.0033127224 * rra_df.mileage
    else:
        rra_df['coef_mlg_x_age'] = np.log((np.exp(0.0001463508 * rra_lob_df.car_age_orig * rra_lob_df.mileage) * np.exp(-0.0004310013 * rra_lob_df.car_age_orig) * np.exp(-0.0033127224 * rra_lob_df.mileage)).mean())
    if leave_out != 'Japanese':
        rra_df['coef_japanese'] = rra_df.car_japanese.map({'Japanese': 0.1004702010, 'Other': 0})
    else:
        rra_df['coef_japanese'] = np.log(rra_lob_df.car_japanese.map({'Japanese': np.exp(0.1004702010), 'Other': np.exp(0)}).mean())
    if leave_out != 'Vehicle class':
        rra_df['coef_class'] = rra_df.car_class_only.map({
            'Compact': 0.0303893787, 'Minivan': np.log(0.975), 'Sporty': 0.0758593720,
            'SUV Large': -0.0237364286, 'SUV Small': 0.0190370746, 'Work Van': 0,
            'Truck': 0.1436074633, 'Car': 0, 'Other': 0})
    else:
        rra_df['coef_class'] = np.log(rra_lob_df.car_class_only.map({
            'Compact': np.exp(0.0303893787), 'Minivan': np.exp(np.log(0.975)), 'Sporty': np.exp(0.0758593720),
            'SUV Large': np.exp(-0.0237364286), 'SUV Small': np.exp(0.0190370746), 'Work Van': np.exp(0),
            'Truck': np.exp(0.1436074633), 'Car': np.exp(0), 'Other': np.exp(0)}).mean())
    if leave_out != 'Fuel type':
        rra_df['coef_fuel'] = rra_df.fuel_type.map({
            'Hybrid': -0.0315481529, 'Flex': 0.0255682747, 'Diesel': -0.1284314925,
            'EV': np.log(0.9), 'Gas': 0})
    else:
        rra_df['coef_fuel'] = np.log(rra_lob_df.fuel_type.map({
            'Hybrid': np.exp(-0.0315481529), 'Flex': np.exp(0.0255682747),
            'Diesel': np.exp(-0.1284314925), 'EV': np.exp(np.log(0.9)), 'Gas': np.exp(0)}).mean())
    if leave_out != 'Luxury classification':
        rra_df['coef_lux'] = rra_df.car_lux.map({'Luxury': -0.1108317813, 'Standard': 0})
    else:
        rra_df['coef_lux'] = np.log(rra_lob_df.car_lux.map({'Luxury': np.exp(-0.1108317813), 'Standard': np.exp(0)}).mean())
    if leave_out != 'LOB':
        rra_df['coef_lob'] = rra_df.lob.map({
            'ENT': 0.0405731662, 'FLD': 0.0405731662, 'FRN': -0.0277901296,
            'STG': 0.0263749504, 'AN': 0})
    else:
        rra_df['coef_lob'] = np.log(rra_lob_df.lob.map({
            'ENT': np.exp(0.0405731662), 'FLD': np.exp(0.0405731662), 'FRN': np.exp(-0.0277901296),
            'STG': np.exp(0.0263749504), 'AN': np.exp(0)}).mean())
    if leave_out != 'Driver flag':
        rra_df['coef_driver'] = (-0.0918049638) * rra_df.driver_flag
    else:
        rra_df['coef_driver'] = np.log(rra_lob_df.driver_flag.map({1: np.exp(-0.0918049638), 0: np.exp(0)}).mean())
    if leave_out != 'Impound probability':
        rra_df['coef_impound'] = (-0.2803330880) * rra_df.impound_prob
    else:
        rra_df['coef_impound'] = np.log(np.mean(np.exp((-0.2803330880) * rra_lob_df.impound_prob)))

    return rra_df.r_mmi * np.exp(
        rra_df.coef_intercept + rra_df.coef_yob + rra_df.coef_mlg_x_age + rra_df.coef_japanese
        + rra_df.coef_class + rra_df.coef_fuel + rra_df.coef_lux + rra_df.coef_lob
        + rra_df.coef_impound + rra_df.coef_driver)


print('ULA multiplier + auc_pred loaded')

ULA multiplier + auc_pred loaded


In [33]:
# =============================================================================
# CELL 5: DATA FETCH (SQL FROM TXT FILES)
# =============================================================================

os.makedirs('cache', exist_ok=True)
force = run_every_query

CACHE_ULA = 'cache/preint_vanillas_ula.pkl'
CACHE_RRA = 'cache/preint_vanillas_rra.pkl'

need_conn = force or not all(
    os.path.exists(p) for p in (CACHE_ULA, CACHE_RRA)
)

if need_conn:
    conn = pyodbc.connect("DSN=Redshift_prod_new")

    with open('preintegration_temptables_query.txt', 'r') as f:
        conn.execute(f.read().strip())
    print('Temp tables created')

    ula_df_total = cached_sql('preintegration_ula_query.txt', CACHE_ULA,
                              connection=conn, force_refresh=force)
    print(f'ULA ready: {len(ula_df_total):,} records')

    rra_df_total = cached_sql('preintegration_rra_query.txt', CACHE_RRA,
                              connection=conn, force_refresh=force)
    print(f'RRA ready: {len(rra_df_total):,} records')

    conn.close()
else:
    ula_df_total = get_pickle(CACHE_ULA)
    rra_df_total = get_pickle(CACHE_RRA)
    print('All data loaded from cache')

print(f"ULA records: {len(ula_df_total):,}")
print(f"RRA records: {len(rra_df_total):,}")
print("[PROGRESS] Data Fetch Complete")

All data loaded from cache
ULA records: 16,493
RRA records: 16,493
[PROGRESS] Data Fetch Complete


In [34]:
# =============================================================================
# CELL 6: DATA PREP + FLAG ENGINEERING
# =============================================================================

# --- Filter and date conversions ---
rra_df_total = rra_df_total[rra_df_total.lob != 'Core']
ula_df_total = ula_df_total[ula_df_total.lob != 'Core']

ula_df_total['app_date'] = pd.to_datetime(ula_df_total['app_date'])
ula_df_total['mtn_3_1_flag'] = ula_df_total.mtn_model == 'MTN3.1'
rra_df_total.app_date = rra_df_total.app_date.astype(str)
ula_df_total.app_date = ula_df_total.app_date.astype(str)
rra_df_total.book_week = rra_df_total.book_week.astype(int).astype(str)
ula_df_total.book_week = ula_df_total.book_week.astype(int).astype(str)

ula_df_total['vehicle_age'] = np.maximum(ula_df_total.app_date.str[:4].astype(int) - ula_df_total.model_year, 1/365)
ula_df_total.tradein_value = ula_df_total.tradein_value.fillna(0)
ula_df_total.make = ula_df_total.make.str.upper().str[:3]
ula_df_total.lob_or_bucket = np.select(
    [ula_df_total.lob_or_bucket.isna() & ula_df_total.lob.isin(['Core', 'FRN']),
     ula_df_total.lob_or_bucket.isna() & ~ula_df_total.lob.isin(['Core', 'FRN'])],
    ['D', 'C'], default=ula_df_total.lob_or_bucket)
ula_df_total['continuous_vehicle_age'] = (
    ula_df_total.app_date.str[:4].astype(int)
    + ula_df_total.app_date.str[5:7].astype(int) / 12
    - (ula_df_total.model_year - 0.25) - 1)

# --- RRA processing ---
rra_df_total['yob'] = expected_years_on_book
rra_df_total['impound_prob'] = impound_probability
rra_df_total.car_make = rra_df_total.car_make.str.upper()
rra_df_total['car_class_only'] = np.select(
    [rra_df_total.vehicle_class.str.contains('Pickup', na=False),
     rra_df_total.vehicle_class.str.contains('Sporty', na=False),
     rra_df_total.vehicle_class.str.contains('Small.*Car', na=False),
     rra_df_total.vehicle_class.str.contains('Car', na=False),
     rra_df_total.vehicle_class.str.contains('Small.*SUV', na=False),
     rra_df_total.vehicle_class.str.contains('Large.*SUV', na=False),
     rra_df_total.vehicle_class.str.contains('Minivan', na=False),
     rra_df_total.vehicle_class.str.contains(' Van', na=False)],
    ['Truck', 'Sporty', 'Compact', 'Car', 'SUV Small', 'SUV Large', 'Minivan', 'Work Van'],
    default='Other')
rra_df_total.fuel_type = np.select(
    [rra_df_total.fuel_type.str.contains('Electric|Plug|EV', na=False),
     rra_df_total.fuel_type.str.contains('Hyb', na=False),
     (rra_df_total.fuel_type == 'CNG') | (rra_df_total.fuel_type == 'LPG'),
     (rra_df_total.fuel_type == 'null') | (rra_df_total.fuel_type == None)],
    ['EV', 'Hybrid', 'Gas', 'Gas'],
    default=rra_df_total.fuel_type)
rra_df_total['car_lux'] = np.where(rra_df_total.vehicle_class.str.contains('Luxury', na=False), 'Luxury', 'Standard')
rra_df_total['car_japanese'] = np.where(
    rra_df_total.car_make.isin(['HONDA', 'ACURA', 'ISUZU', 'MAZDA', 'MITSUBISHI', 'SUZUKI', 'TOYOTA', 'LEXUS', 'SCION', 'TOYOYA']),
    'Japanese', 'Other')

warnings.filterwarnings("ignore", category=UserWarning)
rra_df_total.job_company = rra_df_total.job_company.fillna('not provided')
rra_df_total['driver_flag'] = np.where(
    rra_df_total.job_company.str.contains('(LYFT)|(UBER)|(GRUB ?HUB)|(DOOR ?DASH)|(GO ?PUFF)|(POST ?MATE)|(INSTA ?CART)|(DOMINO)|(PAPA J)|(PIZZA)|(JIMMY ?JOHN)|(SELF)'),
    1, 0)
warnings.filterwarnings("default", category=UserWarning)

# Merge driver_flag from RRA to ULA
ula_df_total = ula_df_total.merge(rra_df_total[['account_number', 'driver_flag']], on='account_number', how='left')

rra_df_total['mileage'] = rra_df_total.mileage_orig / 1000
rra_df_total['r_mmi'] = mmi_standard_increase ** rra_df_total.yob

# --- ULA NA handling ---
ula_df_total = ula_df_total.dropna(subset=['sale_price', 'cd_model_score', 'pti', 'lob'])
ula_df_total.cash_down = ula_df_total.cash_down.fillna(0)
ula_df_total.employment = ula_df_total.employment.fillna('not seasonal or waiter')
ula_df_total.specialty_dealer = ula_df_total.specialty_dealer.fillna('not specialty')
ula_df_total.prev_co_count = ula_df_total.prev_co_count.fillna(0)

# --- ULA filter flags ---
ula_df_total['dealer_filter_a'] = ula_df_total.lob.isin({'AN', 'STG', 'FRN', 'MCY', 'FLD', 'ENT'})
ula_df_total['dealer_filter_b'] = ula_df_total.lob.isin({'AN', 'STG', 'FRN', 'FLD', 'ENT'})

# --- NonKMX Flags ---
ula_df_total['small_amt_financed_flag'] = (ula_df_total.bbvalue < 5000) & (ula_df_total.amt_financed < 4500) & (ula_df_total.lob != 'MCY')
ula_df_total['zero_cash_down_flag'] = (ula_df_total.cash_down <= 250) & (ula_df_total.tradein_value < 3000)
ula_df_total['high_mileage_vehicle_flag'] = (ula_df_total.mileage >= 100000) & (ula_df_total.cd_model_score < 130) & (ula_df_total.lob != 'MCY')
ula_df_total['high_pti_flag'] = (ula_df_total.pti > 0.2) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['low_pti_flag'] = (ula_df_total.pti <= 0.05) & (ula_df_total.cd_model_score >= 130) & ula_df_total.lob.isin({'AN', 'FLD', 'FRN', 'STG'}) & (ula_df_total.cb_flag)
ula_df_total['car_make_penalty_flag'] = ula_df_total.make.isin({'CAD', 'CHR', 'BMW', 'BUI', 'SUB'}) & (ula_df_total.cd_model_score < 130) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_benefit_flag'] = ula_df_total.make.isin({'HON', 'TOY', 'LEX'}) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally') & (ula_df_total.cd_model_score >= 133)
ula_df_total['theft_risk_flag'] = ula_df_total.make.isin({'KIA', 'HYU'}) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally') & (ula_df_total.model_year >= 2015) & (ula_df_total.model_year <= 2021) & (ula_df_total.app_date >= '2022-07-01')
ula_df_total['mcy_low_mileage_flag'] = (ula_df_total.lob == 'MCY') & (ula_df_total.cd_model_score > 140) & (ula_df_total.mileage <= 20000) & (ula_df_total.vehicle_age <= 10)
ula_df_total['weekend_flag'] = ula_df_total.day_of_week.isin([0, 6])
ula_df_total['weekday_flag'] = ula_df_total.day_of_week.isin(range(1, 6))
ula_df_total['secured_credit_flag'] = ula_df_total.secured_credit_card
ula_df_total['chime_flag'] = ula_df_total.chime_indicator
ula_df_total['nonkmx_chime_flag'] = ula_df_total.nonkmx_chime_indicator
ula_df_total['seasonal_employment_flag'] = ula_df_total.employment == 'seasonal'
ula_df_total['waiter_employment_flag'] = ula_df_total.employment == 'waiter'
ula_df_total['nonkmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines > 0.2
ula_df_total['prev_co_flag'] = ula_df_total.prev_co_count > 0
ula_df_total['null_fico_w_vantage_flag'] = ((ula_df_total.fico_score < 300) | (ula_df_total.fico_score > 850)) & (ula_df_total.vantage_score >= 300) & (ula_df_total.vantage_score <= 850)
ula_df_total['null_fico_null_vantage_flag'] = ((ula_df_total.fico_score < 300) | (ula_df_total.fico_score > 850)) & ((ula_df_total.vantage_score < 300) | (ula_df_total.vantage_score > 850))
ula_df_total['pricing_change_flag'] = ula_df_total.book_date >= pd.to_datetime('2024-10-01').date()
ula_df_total['student_loans_cutoff_date'] = ula_df_total.book_date >= pd.to_datetime('2023-05-01').date()
ula_df_total['ent_fld_flag'] = (ula_df_total.lob == 'ENT') | (ula_df_total.lob == 'FLD')
ula_df_total['ent_flag'] = (ula_df_total.lob == 'ENT')
ula_df_total['student_loan_flag'] = 0

# --- KMX Flags (carried for completeness) ---
ula_df_total['job_time_flag'] = ula_df_total.employed_months < 6
ula_df_total['low_bureau_flag'] = ((ula_df_total.fico_score > 300) & (ula_df_total.fico_score < 475)) | (ula_df_total.vantage_score > 4) & ((ula_df_total.vantage_score < 450))
ula_df_total['low_fico_flag'] = (ula_df_total.fico_score > 300) & (ula_df_total.fico_score < 450)
ula_df_total['high_model_score_flag'] = ula_df_total.cd_model_score >= 146
ula_df_total['low_vantage_flag'] = (ula_df_total.fico_score < 300) & (ula_df_total.vantage_score > 4) & (ula_df_total.vantage_score < 450)
ula_df_total['louisiana_flag'] = ula_df_total.state == 'LA'
ula_df_total['normal_pti_flag'] = ula_df_total.pti <= 0.2
ula_df_total['high_pti_tier_1_flag'] = (ula_df_total.pti > 0.2) & (ula_df_total.pti <= 0.25)
ula_df_total['high_pti_tier_2_flag'] = (ula_df_total.pti > 0.25) & (ula_df_total.pti <= 0.35)
ula_df_total['high_pti_tier_3_flag'] = ula_df_total.pti > 0.35
ula_df_total['existing_dq_flag'] = ula_df_total.existing_dq_count > 0
ula_df_total['kmx_toyho_flag'] = ula_df_total.cd_model_score >= 130 & ula_df_total.make.isin({'HON', 'TOY'})
ula_df_total['kmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines >= 0.14
ula_df_total['soft_pull_flag'] = ula_df_total.pull_type == 'softpull'
ula_df_total['cd_perc_flag'] = (ula_df_total.sale_price <= 18000) & (ula_df_total.cash_down / ula_df_total.sale_price <= 0.1)
ula_df_total['open_tl_flag'] = ula_df_total.open_tl == 0
ula_df_total['narrowed_soft_pull_flag'] = (ula_df_total.sale_price <= 18000) & (ula_df_total.cash_down / ula_df_total.sale_price <= 0.1) & ula_df_total.soft_pull_flag & (~ula_df_total.job_time_flag)

# --- Vintage assignment ---
rra_df_total['vintage'] = rra_df_total.app_date.str[:4] + ' M' + rra_df_total.app_date.str[5:7]
ula_df_total['vintage'] = ula_df_total.app_date.str[:4] + ' M' + ula_df_total.app_date.str[5:7]

# --- Employment dedup ---
ula_df_total = ula_df_total.drop_duplicates().copy()
ula_df_total['employment_type_code'] = ula_df_total.seasonal_employment_flag * 1 + ula_df_total.waiter_employment_flag * 8
combined_employment_code_df = ula_df_total.groupby('account_number').employment_type_code.sum().reset_index()
ula_df_total = ula_df_total.drop(columns='employment_type_code').merge(combined_employment_code_df, on='account_number')
ula_df_total.seasonal_employment_flag = (ula_df_total.employment_type_code == 1) | (ula_df_total.employment_type_code == 9)
ula_df_total.waiter_employment_flag = (ula_df_total.employment_type_code == 8) | (ula_df_total.employment_type_code == 9)
ula_df_total = ula_df_total.drop(columns='employment').drop_duplicates()

# --- RRA dedup ---
combined_driver_flag_df = rra_df_total.groupby('account_number').driver_flag.max().reset_index()
rra_df_total = rra_df_total.drop(columns='driver_flag').merge(combined_driver_flag_df, on='account_number')
rra_df_total = rra_df_total.drop(columns='job_company').drop_duplicates()

print(f"ULA after refinement: {len(ula_df_total):,}")
print(f"RRA after refinement: {len(rra_df_total):,}")

# --- Pool population counts ---
for pool_name, pool_cfg in POOLS.items():
    n = len(pool_cfg['filter'](ula_df_total)) if pool_cfg['filter'] else len(ula_df_total)
    print(f"  {pool_name}: ULA={n:,}")

print("[PROGRESS] Data Prep Complete")

ULA after refinement: 16,492
RRA after refinement: 16,493
  Vanilla: ULA=16,492
  Downcredit: ULA=15,881
  Upcredit: ULA=611
[PROGRESS] Data Prep Complete


In [35]:
# =============================================================================
# CELL 7: POOL-LEVEL RAGU SCORING
# =============================================================================

def get_ragu_score(vintage, lob, ula_df_sub, rra_df_sub, ms_df, baseline_config, leave_out='None'):
    baseline_ltv = baseline_config['ltv']
    baseline_recovery_pct = baseline_config['recovery_pct']
    baseline_recovery_unadjusted_pct = baseline_config['recovery_unadjusted_pct']
    baseline_apr = baseline_config['apr']

    ula_df = ula_df_sub[(ula_df_sub.vintage == vintage) & (ula_df_sub.lob == lob)].copy()
    if len(ula_df) == 0:
        return None

    ula_df = get_ula_multiplier_nonkmx(ula_df, leave_out)
    ula_df = ula_df[['account_number', 'app_date', 'bbvalue', 'sale_price', 'amt_financed',
                     'lob_or_bucket', 'lob', 'loss_multiplier', 'apr']]

    rra_df = rra_df_sub[(rra_df_sub.vintage == vintage) & (rra_df_sub.lob == lob)].copy()
    try:
        rra_df.car_year = rra_df.car_year.fillna(round(rra_df.car_year.mean()))
    except:
        pass
    rra_df['car_age_orig'] = np.maximum(
        (pd.to_datetime(rra_df.app_date) - pd.to_datetime(rra_df.car_year.astype(int).astype(str) + '-09-01')).dt.days / 365,
        1 / 365)
    rra_df['recovery_multiplier'] = auc_pred(rra_df, leave_out)

    rra_df = rra_df[['account_number', 'recovery_multiplier']]
    mix_df = ula_df.merge(rra_df, on='account_number', how='left').drop_duplicates(subset='account_number', keep='first')

    full_pop_metrics = mix_df.groupby('lob').apply(
        weighted_average_and_sum,
        ['loss_multiplier', 'apr'],
        include_groups=False
    )

    bb_populated_df = mix_df[mix_df['bbvalue'].notna() & (mix_df['bbvalue'] > 0)].copy()
    if len(bb_populated_df) > 0:
        bb_populated_df['ltv'] = bb_populated_df.amt_financed / bb_populated_df.bbvalue
        bb_pop_metrics = bb_populated_df.groupby('lob').apply(
            weighted_average_and_sum,
            ['ltv', 'bbvalue'],
            include_groups=False
        ).drop(columns='amt_financed')
    else:
        bb_pop_metrics = pd.DataFrame(columns=['ltv', 'bbvalue'])

    recovery_df = mix_df[
        mix_df['bbvalue'].notna() & (mix_df['bbvalue'] > 0) & mix_df['recovery_multiplier'].notna()
    ].copy()
    if len(recovery_df) > 0:
        recovery_df['recovery_unadjusted_multiplier'] = skip_rate * recovery_df.recovery_multiplier
        recovery_metrics = recovery_df.groupby('lob').apply(
            weighted_average_and_sum,
            ['recovery_unadjusted_multiplier'],
            include_groups=False
        ).drop(columns='amt_financed')
    else:
        recovery_metrics = pd.DataFrame(columns=['recovery_unadjusted_multiplier'])

    grouped_mix_df = full_pop_metrics.join(bb_pop_metrics).join(recovery_metrics)
    grouped_mix_df['recovery_unadjusted_multiplier'] = grouped_mix_df['recovery_unadjusted_multiplier'].fillna(0)
    grouped_mix_df['ltv'] = grouped_mix_df['ltv'].fillna(baseline_ltv)
    grouped_mix_df['bbvalue'] = grouped_mix_df['bbvalue'].fillna(0)
    grouped_mix_df['ltv_realization_factor'] = 0
    grouped_mix_df['recovery_multiplier'] = grouped_mix_df.recovery_unadjusted_multiplier * (
        1 + (baseline_ltv / grouped_mix_df.ltv - 1) * grouped_mix_df.ltv_realization_factor)

    vintage_ms_df = ms_df[ms_df.vintage == vintage][['lob', 'model_score']].copy()

    index_name = grouped_mix_df.index.name
    if isinstance(index_name, str) and index_name in grouped_mix_df.columns:
        grouped_mix_df = grouped_mix_df.reset_index(drop=True)
    else:
        grouped_mix_df = grouped_mix_df.reset_index()

    full_df = grouped_mix_df.merge(vintage_ms_df, on='lob')
    if len(full_df) == 0:
        return None

    full_df['est_unit_loss'] = mean_unit_loss
    full_df['unit_loss_score'] = full_df.model_score + (1 - full_df.loss_multiplier) * full_df.est_unit_loss / unit_loss_to_model_score
    full_df = full_df.set_index('lob')

    full_df['ms_original'] = full_df.model_score.copy()
    full_df['baselined_recovery'] = (full_df.recovery_multiplier / baseline_recovery_pct).copy()
    full_df['baselined_unadjusted_recovery'] = (full_df.recovery_unadjusted_multiplier / baseline_recovery_unadjusted_pct).copy()

    full_df['gross_loss_impact'] = full_df['unit_loss_score'] - full_df['ms_original']
    full_df['ltv_impact'] = ((baseline_ltv / full_df['ltv']) - 1) * 17
    full_df['recovery_impact'] = full_df['unit_loss_score'] * full_df['est_unit_loss'] * full_df['recovery_multiplier'] * (full_df['baselined_recovery'] - 1)
    full_df['apr_impact'] = (baseline_apr - full_df['apr']) * 70
    full_df['ragu_score'] = (
        (1 - full_df.est_unit_loss * full_df.recovery_multiplier) * full_df.unit_loss_score
        + full_df.est_unit_loss * full_df.recovery_multiplier * full_df.unit_loss_score * full_df.baselined_recovery
        + full_df['ltv_impact']
        + full_df['apr_impact'])

    full_df['vintage'] = vintage
    return full_df


# --- Main loop: iterate over pools ---
all_vintages = sorted(ula_df_total['vintage'].unique())
all_pool_results = []

for pool_name, pool_cfg in POOLS.items():
    ula_subset = pool_cfg['filter'](ula_df_total) if pool_cfg['filter'] else ula_df_total.copy()

    if len(ula_subset) == 0:
        print(f'{pool_name}: no ULA loans found, skipping')
        continue

    ms_subset = ula_subset.groupby(['vintage', 'lob']).apply(
        weighted_average_and_sum, 'cd_model_score', include_groups=False
    ).reset_index()
    ms_subset = ms_subset.rename(columns={'cd_model_score': 'model_score'})

    baseline_config = BASELINES['STG']
    lob_results = []

    for vintage in all_vintages:
        try:
            result = get_ragu_score(
                vintage, 'STG', ula_subset, rra_df_total, ms_subset, baseline_config)
            if result is not None:
                lob_results.append(result)
        except Exception as e:
            print(f'  Error: {pool_name} / {vintage}: {e}')

    if not lob_results:
        print(f'{pool_name}: no scoreable vintages')
        continue

    pool_df = pd.concat(lob_results, ignore_index=False).reset_index()

    rollup_metrics = [
        'ms_original', 'gross_loss_impact', 'recovery_impact',
        'ltv_impact', 'apr_impact', 'ragu_score', 'ltv', 'apr',
    ]
    rollup = pool_df.groupby('vintage').apply(
        weighted_average_and_sum, rollup_metrics, include_groups=False
    ).reset_index()
    rollup['lob'] = pool_name
    all_pool_results.append(rollup)

    n_vintages = rollup.vintage.nunique()
    print(f'{pool_name}: {n_vintages} vintages scored ({len(ula_subset):,} ULA loans)')

all_df = pd.concat(all_pool_results, ignore_index=True)
print(f'\nTotal results: {len(all_df)} rows across {all_df.lob.nunique()} pools and {all_df.vintage.nunique()} vintages')
print('[PROGRESS] Pool RAGU Scoring Complete')

Vanilla: 17 vintages scored (16,492 ULA loans)
Downcredit: 17 vintages scored (15,881 ULA loans)
Upcredit: 17 vintages scored (611 ULA loans)

Total results: 51 rows across 3 pools and 17 vintages
[PROGRESS] Pool RAGU Scoring Complete


In [36]:
# =============================================================================
# CELL 8: DISPLAY + EXCEL EXPORT
# =============================================================================

METRIC_ROWS = [
    ('Model Score',       'ms_original'),
    ('Gross Loss Impact', 'gross_loss_impact'),
    ('Recovery Impact',   'recovery_impact'),
    ('LTV Impact',        'ltv_impact'),
    ('APR Impact',        'apr_impact'),
    ('RAGU Score',        'ragu_score'),
    ('Amount Financed',   'amt_financed'),
    ('Weighted LTV',      'ltv'),
    ('Weighted APR',      'apr'),
]

EXCEL_OUTPUT = 'preintegration_ragu_vanillas.xlsx'
sheet_name = 'Data Tables (M)'

sorted_vintages = sorted(all_df['vintage'].unique())
all_export_pools = list(POOLS.keys())

# --- Display summary per pool ---
pd.set_option('display.float_format', '{:.4f}'.format)
for pool_name in all_export_pools:
    pool_data = all_df[all_df.lob == pool_name]
    if len(pool_data) == 0:
        continue
    print(f'\n=== {pool_name} ===')
    pivot = pool_data.set_index('vintage')[['ms_original', 'gross_loss_impact',
        'recovery_impact', 'ltv_impact', 'apr_impact', 'ragu_score', 'amt_financed']].T
    display(pivot)

# --- Excel export ---
if os.path.exists(EXCEL_OUTPUT):
    wb = openpyxl.load_workbook(EXCEL_OUTPUT)
    if sheet_name in wb.sheetnames:
        del wb[sheet_name]
    ws = wb.create_sheet(sheet_name)
else:
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = sheet_name

current_row = 1
for pool_name in all_export_pools:
    pool_data = all_df[all_df.lob == pool_name].set_index('vintage')
    if len(pool_data) == 0:
        continue

    ws.cell(row=current_row, column=1, value=pool_name)
    for col_idx, v in enumerate(sorted_vintages, start=2):
        ws.cell(row=current_row, column=col_idx, value=v)
    current_row += 1

    for label, col_key in METRIC_ROWS:
        ws.cell(row=current_row, column=1, value=label)
        for col_idx, v in enumerate(sorted_vintages, start=2):
            if v in pool_data.index:
                ws.cell(row=current_row, column=col_idx, value=pool_data.loc[v, col_key])
        current_row += 1
    current_row += 1

wb.save(EXCEL_OUTPUT)
print(f'\nSaved to {EXCEL_OUTPUT} (sheet: {sheet_name})')
print(f'  {len(all_export_pools)} pools x {len(sorted_vintages)} periods')
print('[PROGRESS] Excel Export Complete')


=== Vanilla ===


vintage,2024 M06,2024 M07,2024 M08,2024 M09,2024 M10,2024 M11,2024 M12,2025 M01,2025 M02,2025 M03,2025 M04,2025 M05,2025 M06,2025 M07,2025 M08,2025 M09,2025 M10
ms_original,129.8654,129.3877,129.5371,129.5132,129.6633,129.2518,129.6836,129.5373,129.3933,129.7778,129.7131,129.6155,129.4454,129.2781,129.2425,128.5093,128.3454
gross_loss_impact,1.2350,0.8490,0.7009,0.5628,0.8645,0.8530,0.8039,-0.3715,-1.3313,-1.3310,-1.0314,-0.9627,-1.1900,-0.5360,0.7759,0.6386,0.7673
recovery_impact,-0.0000,-0.0000,4.1386,2.1729,1.2089,2.3985,2.8808,2.2101,0.2182,0.8428,1.1158,0.6396,0.7012,0.9091,1.9626,0.6936,0.5372
ltv_impact,0.0000,0.0000,5.2740,6.0441,5.7001,4.9096,5.2506,5.7306,5.4984,5.5410,6.3776,5.3168,4.7133,4.4568,4.2806,3.8856,3.3647
apr_impact,3.0477,3.3127,2.9734,2.2654,2.0309,2.2986,2.3220,2.0590,1.8285,1.8606,2.0087,2.0073,2.1051,2.1983,2.0738,2.0963,1.9258
ragu_score,134.1481,133.5494,142.6239,140.5584,139.4677,139.7115,140.9409,139.1655,135.6071,136.6912,138.1838,136.6165,135.7751,136.3063,138.3354,135.8234,134.9403
amt_financed,1462434.4300,10919535.7800,18319871.1800,17387363.6500,23124191.2000,23510720.6400,31543002.8400,35545767.2600,50802743.6800,55019552.1600,42442574.0100,40719961.4000,38242116.3900,43485809.9400,43419978.6100,44626995.3800,9686023.1400



=== Downcredit ===


vintage,2024 M06,2024 M07,2024 M08,2024 M09,2024 M10,2024 M11,2024 M12,2025 M01,2025 M02,2025 M03,2025 M04,2025 M05,2025 M06,2025 M07,2025 M08,2025 M09,2025 M10
ms_original,129.6258,129.1947,128.9105,129.1747,128.9294,128.7007,129.0721,128.7343,128.7927,129.1737,129.1948,128.9273,128.7038,128.5785,128.4349,127.9708,127.7657
gross_loss_impact,1.2323,0.8351,0.6503,0.5428,0.8300,0.8324,0.7565,-0.4525,-1.4331,-1.4058,-1.0807,-1.0532,-1.3189,-0.5718,0.7480,0.6116,0.7243
recovery_impact,-0.0000,-0.0000,4.1171,2.1669,1.2238,2.2756,2.8874,1.9501,0.1773,0.9004,1.1560,0.6357,0.6982,0.9229,1.9808,0.6808,0.4731
ltv_impact,0.0000,0.0000,5.2740,6.0441,5.7074,4.5610,5.0284,5.6549,5.4040,5.4122,6.1336,5.3168,4.5216,4.2179,4.0557,3.7435,3.0913
apr_impact,3.0305,3.3146,2.9512,2.2435,1.9667,2.2701,2.2876,1.9899,1.7821,1.8123,1.9794,1.9434,2.0484,2.1726,2.0152,2.0694,1.8813
ragu_score,133.8886,133.3444,141.9030,140.1720,138.6573,138.6398,140.0320,137.8767,134.7230,135.8928,137.3830,135.7699,134.6531,135.3201,137.2347,135.0761,133.9358
amt_financed,1441039.4300,10755488.8600,17478642.4000,16953632.0700,21831384.9200,22627227.3800,30132247.0200,33659930.1700,48601529.7900,52571150.7900,40861968.5200,38735113.1900,36242929.7600,41457316.4500,41077113.2300,43058949.3400,9300708.1300



=== Upcredit ===


vintage,2024 M06,2024 M07,2024 M08,2024 M09,2024 M10,2024 M11,2024 M12,2025 M01,2025 M02,2025 M03,2025 M04,2025 M05,2025 M06,2025 M07,2025 M08,2025 M09,2025 M10
ms_original,146.0000,142.0442,142.5560,142.7442,142.0565,143.3644,142.7451,143.8704,142.6547,142.7484,143.1126,143.0467,142.8893,143.5753,143.4027,143.2948,142.3374
gross_loss_impact,1.4166,1.7640,1.7523,1.3410,1.4477,1.3813,1.8172,1.0755,0.9157,0.2750,0.2441,0.8042,1.1473,0.1951,1.2646,1.3804,1.8049
recovery_impact,-0.0000,-0.0000,-0.0000,-0.0000,0.4252,4.5945,2.6684,8.9723,1.8306,-0.8324,0.0127,-0.0000,0.7327,0.5658,1.4560,1.1348,2.3805
ltv_impact,0.0000,0.0000,0.0000,0.0000,5.4272,11.7050,11.7970,7.3978,9.2272,9.6832,14.8890,0.0000,11.9129,11.2730,10.6163,9.7696,13.5679
apr_impact,4.2070,3.1819,3.4343,3.1233,3.1148,3.0286,3.0568,3.2916,2.8530,2.8971,2.7658,3.2562,3.1338,2.7236,3.1012,2.8370,3.0003
ragu_score,151.6236,146.9901,147.7426,147.2085,152.4714,164.0738,162.0845,164.6076,157.4811,154.7713,161.0243,147.1071,159.8161,158.3328,159.8408,158.4166,163.0910
amt_financed,21395.0000,164046.9200,841228.7800,433731.5800,1292806.2800,883493.2600,1410755.8200,1885837.0900,2201213.8900,2448401.3700,1580605.4900,1984848.2100,1999186.6300,2028493.4900,2342865.3800,1568046.0400,385315.0100



Saved to preintegration_ragu_vanillas.xlsx (sheet: Data Tables (M))
  3 pools x 17 periods
[PROGRESS] Excel Export Complete


In [37]:
# =============================================================================
# CELL 9: DIAGNOSTIC -- POPULATION FUNNEL (LEFT-JOIN FLOW)
# =============================================================================
# Reflects the updated scoring flow:
#   mix_df (left merge ULA+RRA) -> full_pop_metrics (loss_multiplier, apr)
#   bb_populated_df (bbvalue > 0) -> bb_pop_metrics (ltv, bbvalue)
#   recovery_df (bb + recovery.notna()) -> recovery_metrics

ula_raw = get_pickle(CACHE_ULA)
rra_raw = get_pickle(CACHE_RRA)

print("=== RAW DATA FROM SQL ===")
print(f"ULA raw records: {len(ula_raw):,}")
print(f"RRA raw records: {len(rra_raw):,}")
print(f"ULA unique accounts: {ula_raw.account_number.nunique():,}")
print(f"RRA unique accounts: {rra_raw.account_number.nunique():,}")

ula_raw['app_date_str'] = pd.to_datetime(ula_raw['app_date']).astype(str)
ula_raw['vintage'] = ula_raw['app_date_str'].str[:4] + ' M' + ula_raw['app_date_str'].str[5:7]

rra_raw['app_date_str'] = pd.to_datetime(rra_raw['app_date']).astype(str)
rra_raw['vintage'] = rra_raw['app_date_str'].str[:4] + ' M' + rra_raw['app_date_str'].str[5:7]

vintages = sorted(ula_raw['vintage'].unique())
rows = []

for v in vintages:
    ula_v = ula_raw[ula_raw.vintage == v]
    rra_v = rra_raw[rra_raw.vintage == v]

    n_ula_raw = ula_v.account_number.nunique()
    n_rra_raw = rra_v.account_number.nunique()

    null_sale_price = ula_v.sale_price.isna().sum()
    null_model_score = ula_v.cd_model_score.isna().sum()
    null_pti = ula_v.pti.isna().sum()
    ula_after_dropna = ula_v.dropna(subset=['sale_price', 'cd_model_score', 'pti', 'lob'])
    n_after_dropna = ula_after_dropna.account_number.nunique()

    ula_accts = set(ula_after_dropna.account_number.unique())
    rra_accts = set(rra_v.account_number.unique())
    with_recovery = ula_accts & rra_accts
    without_recovery = ula_accts - rra_accts

    bb_null = ula_after_dropna.bbvalue.isna().sum()
    bb_zero = (ula_after_dropna.bbvalue == 0).sum()
    bb_positive = (ula_after_dropna.bbvalue > 0).sum()
    n_bb_accts = ula_after_dropna[
        ula_after_dropna.bbvalue.notna() & (ula_after_dropna.bbvalue > 0)
    ].account_number.nunique()

    bb_and_recovery = ula_after_dropna[
        ula_after_dropna.bbvalue.notna()
        & (ula_after_dropna.bbvalue > 0)
        & ula_after_dropna.account_number.isin(with_recovery)
    ].account_number.nunique()

    rows.append({
        'vintage': v,
        'ula_raw': n_ula_raw,
        'rra_raw': n_rra_raw,
        'null_sale_price': null_sale_price,
        'null_model_score': null_model_score,
        'null_pti': null_pti,
        'after_dropna': n_after_dropna,
        'dropped_by_dropna': n_ula_raw - n_after_dropna,
        'full_pop_for_lm_apr': n_after_dropna,
        'with_recovery': len(with_recovery),
        'without_recovery': len(without_recovery),
        'bb_null': bb_null,
        'bb_zero': bb_zero,
        'bb_positive': bb_positive,
        'bb_pop_for_ltv': n_bb_accts,
        'bb_and_recovery': bb_and_recovery,
        'pct_full_pop': f"{n_after_dropna / n_ula_raw * 100:.1f}%" if n_ula_raw > 0 else "N/A",
        'pct_bb_pop': f"{n_bb_accts / n_ula_raw * 100:.1f}%" if n_ula_raw > 0 else "N/A",
        'pct_recovery_pop': f"{bb_and_recovery / n_ula_raw * 100:.1f}%" if n_ula_raw > 0 else "N/A",
    })

funnel = pd.DataFrame(rows).set_index('vintage')
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.0f}'.format)

print("\n=== PER-VINTAGE FUNNEL (unique accounts) ===")
display(funnel)

total_raw = funnel['ula_raw'].sum()
total_dropna_loss = funnel['dropped_by_dropna'].sum()
total_full_pop = funnel['full_pop_for_lm_apr'].sum()
total_without_recovery = funnel['without_recovery'].sum()
total_bb_pop = funnel['bb_pop_for_ltv'].sum()
total_recovery_pop = funnel['bb_and_recovery'].sum()

print("\n=== SUMMARY: POPULATION BY METRIC PATH ===")
print(f"  Raw ULA accounts:            {total_raw:,}")
print(f"  Lost to dropna:              {total_dropna_loss:,} ({total_dropna_loss/total_raw*100:.1f}%)")
print(f"  Full pop (loss_mult + APR):  {total_full_pop:,} ({total_full_pop/total_raw*100:.1f}%)")
print(f"    - with recovery data:      {total_full_pop - total_without_recovery:,}")
print(f"    - without recovery data:   {total_without_recovery:,}")
print(f"  BB pop (LTV + bbvalue):      {total_bb_pop:,} ({total_bb_pop/total_raw*100:.1f}%)")
print(f"  BB + recovery (recovery):    {total_recovery_pop:,} ({total_recovery_pop/total_raw*100:.1f}%)")

=== RAW DATA FROM SQL ===
ULA raw records: 16,493
RRA raw records: 16,493
ULA unique accounts: 16,493
RRA unique accounts: 16,493



=== PER-VINTAGE FUNNEL (unique accounts) ===


,ula_raw,rra_raw,null_sale_price,null_model_score,null_pti,after_dropna,dropped_by_dropna,full_pop_for_lm_apr,with_recovery,without_recovery,bb_null,bb_zero,bb_positive,bb_pop_for_ltv,bb_and_recovery,pct_full_pop,pct_bb_pop,pct_recovery_pop
vintage,,,,,,,,,,,,,,,,,,
2024 M06,46,46,0,0,0,46,0,46,46,0,46,0,0,0,0,100.0%,0.0%,0.0%
2024 M07,286,286,0,0,0,286,0,286,286,0,286,0,0,0,0,100.0%,0.0%,0.0%
2024 M08,498,498,0,0,0,498,0,498,498,0,485,0,13,13,13,100.0%,2.6%,2.6%
2024 M09,544,544,0,0,0,544,0,544,544,0,514,0,30,30,30,100.0%,5.5%,5.5%
2024 M10,758,758,0,0,0,758,0,758,758,0,694,0,64,64,64,100.0%,8.4%,8.4%
2024 M11,737,737,0,1,0,736,1,736,736,0,648,0,88,88,88,99.9%,11.9%,11.9%
2024 M12,962,962,0,0,0,962,0,962,962,0,858,0,104,104,104,100.0%,10.8%,10.8%
2025 M01,1121,1121,0,0,0,1121,0,1121,1121,0,1027,0,94,94,94,100.0%,8.4%,8.4%
2025 M02,1690,1690,0,0,0,1690,0,1690,1690,0,1384,0,306,306,306,100.0%,18.1%,18.1%



=== SUMMARY: POPULATION BY METRIC PATH ===
  Raw ULA accounts:            16,493
  Lost to dropna:              1 (0.0%)
  Full pop (loss_mult + APR):  16,492 (100.0%)
    - with recovery data:      16,492
    - without recovery data:   0
  BB pop (LTV + bbvalue):      5,463 (33.1%)
  BB + recovery (recovery):    5,463 (33.1%)
